# RPM Fraud Analysis -- Archived Methodology Tests

Peer grouping evaluation cells extracted from rpm_fraud_analysis.ipynb.
These ran after cells 1-7 of the main notebook.

**Result**: Global z-scores selected. No peer grouping reduces within-group variance meaningfully.
See Work Vault RPM.md entry 2026-09-03c for full writeup.

## Prerequisites

Run cells 1-7 of rpm_fraud_analysis.ipynb first, then copy df, z_features, sql_signals, compute_z.

### 2a. Peer grouping evaluation (archived)

Evaluated four strategies. **Result: global (no peers) selected.** Specialty-based peer grouping reduced variance by only 6.5%. Size-based was worse than global (-18.4%). No grouping variable explains RPM billing behavior because RPM is a mechanistic service with the same CPT codes regardless of provider specialty.

Strategies tested:
1. **Global (no peers)** -- all TINs in one pool (selected)
2. **Size-based** -- bucket by member count (25-74, 75-199, 200-499, 500+)
3. **Practice-type** -- CMS specialty mapped to 5 broad categories
4. **Specialty (previous)** -- raw CMS specialty code, 30-TIN minimum

In [83]:
## --- Define the three grouping strategies ---

# Size buckets: based on total_mbrs
df["size_bucket"] = pd.cut(
    df.total_mbrs,
    bins = [0, 74, 199, 499, float("inf")],
    labels = ["25-74", "75-199", "200-499", "500+"],
    right = True
)

# Practice-type mapping from CMS specialty codes
# Primary care: 1 (GP), 8 (Family), 11 (Internal Med), 37 (Pediatric), 38 (Geriatric)
# Cardio/Pulm: 6 (Cardiovascular), 29 (Pulmonary)
# NP/PA: 50 (NP), 89 (CNS), 97 (PA)
# Home health/Facility: 72 (Home Health), 22 (Pathology), 49-52 (Hospital-based)
# Other: everything else
def map_practice_type(spec):
    if pd.isna(spec):
        return "Other"
    s = int(spec)
    if s in (1, 8, 11, 37, 38):
        return "Primary Care"
    if s in (6, 29):
        return "Cardio/Pulm"
    if s in (50, 89, 97):
        return "NP/PA"
    if s in (22, 49, 51, 52, 72):
        return "Facility/Home Health"
    return "Other"

df["practice_type"] = df.primary_specialty.apply(map_practice_type)

# Check group sizes for each strategy
print("Size bucket distribution:")
print(df.query("total_mbrs >= 25").size_bucket.value_counts().sort_index().to_string())
print(f"\nPractice type distribution:")
print(df.practice_type.value_counts().to_string())
spec_sizes = df.groupby("primary_specialty").size()
print(f"\nSpecialty distribution (current):")
print(f"Groups with 30+ TINs: {(spec_sizes >= 30).sum()}")
valid_specs = spec_sizes[spec_sizes >= 30].index
print(f"TINs in valid groups: {df.primary_specialty.isin(valid_specs).sum()} / {len(df)}")

Size bucket distribution:
size_bucket
25-74      817
75-199     302
200-499     97
500+        44

Practice type distribution:
practice_type
Other                   3844
Primary Care            1991
Facility/Home Health     265
Cardio/Pulm               74
NP/PA                     38

Specialty distribution (current):
Groups with 30+ TINs: 17
TINs in valid groups: 5905 / 6212


In [84]:
## --- Compute z-scores under each strategy ---

MIN_GROUP = 30  # minimum group size to use within-group z-scores

def compute_z(series):
    return (series - series.mean()) / max(series.std(), 1e-10)

def compute_z_by_strategy(df, z_features, group_col=None, min_group=MIN_GROUP):
    """Compute z-scores. If group_col is None, use global. Otherwise group with fallback."""
    z_cols = {}
    for feat in z_features:
        col = f"z_{feat}"
        # start with global z for everyone
        z_cols[col] = compute_z(df[feat])
        if group_col is not None:
            gsizes = df.groupby(group_col).size()
            valid_groups = gsizes[gsizes >= min_group].index
            for grp_val, grp_df in df[df[group_col].isin(valid_groups)].groupby(group_col):
                idx = grp_df.index
                z_cols[col].loc[idx] = compute_z(grp_df[feat])
    return pd.DataFrame(z_cols, index=df.index)

strategies = {
    "Global (no peers)": None,
    "Size-based": "size_bucket",
    "Practice-type": "practice_type",
    "Specialty (current)": "primary_specialty",
}

results = {}
for name, group_col in strategies.items():
    z_df = compute_z_by_strategy(df, z_features, group_col)
    n_z_flags = z_df.gt(3).sum(axis=1)
    flagged = (n_z_flags >= 2).astype(int)

    # variance reduction: avg within-group std / global std
    if group_col is not None:
        vr_list = []
        for feat in z_features:
            global_std = df[feat].std()
            gsizes = df.groupby(group_col).size()
            valid = gsizes[gsizes >= MIN_GROUP].index
            grp_stds = df[df[group_col].isin(valid)].groupby(group_col)[feat].std()
            if global_std > 0 and len(grp_stds) > 0:
                vr_list.append(1 - grp_stds.mean() / global_std)
        avg_vr = np.mean(vr_list)
    else:
        avg_vr = 0.0  # global is the baseline

    results[name] = {
        "z_df": z_df,
        "n_z_flags": n_z_flags,
        "flagged": flagged,
        "avg_variance_reduction": avg_vr,
        "n_flagged": flagged.sum(),
        "n_any_z3": z_df.gt(3).any(axis=1).sum(),
    }

print(f"{'Strategy':<25} {'Var Reduction':>14} {'Any z>3':>10} {'Flagged (2+)':>14}")
print("-" * 67)
for name, r in results.items():
    print(f"{name:<25} {r['avg_variance_reduction']:>13.1%} {r['n_any_z3']:>10} {r['n_flagged']:>14}")

Strategy                   Var Reduction    Any z>3   Flagged (2+)
-------------------------------------------------------------------
Global (no peers)                  0.0%        613             87
Size-based                       -18.4%        829            106
Practice-type                      4.8%       1034            161
Specialty (current)                6.5%       1106            198


In [ ]:
## --- Overlap with SQL signals ---
## If a peer strategy is good, the TINs it flags should also have corroborating SQL signals.
## A strategy that flags TINs with zero other signals is finding noise.

sql_sig_cols = ["sig_no_prior_rel", "sig_ramp", "sig_billing_reg", "sig_coding",
                "sig_churn", "sig_geo", "sig_sequencing", "sig_linkage"]

print(f"{'Strategy':<25} {'Flagged':>8} {'Mean SQL sigs':>14} {'0 SQL sigs':>11} {'3+ SQL sigs':>12}")
print("-" * 74)
for name, r in results.items():
    flagged_mask = r["flagged"] == 1
    if flagged_mask.sum() == 0:
        print(f"{name:<25} {'0':>8}")
        continue
    flagged_df = df[flagged_mask]
    sql_totals = flagged_df[sql_sig_cols].sum(axis=1)
    mean_sql = sql_totals.mean()
    zero_sql = (sql_totals == 0).sum()
    three_plus = (sql_totals >= 3).sum()
    print(f"{name:<25} {r['n_flagged']:>8} {mean_sql:>14.2f} {zero_sql:>11} {three_plus:>12}")

In [ ]:
## --- Vendor calibration ---
## Known/suspected vendors (vendor_name_flag == 1) should be flagged.
## A strategy that misses known vendors is too conservative.

vendor_mask = df.vendor_name_flag == 1
n_vendors = vendor_mask.sum()

print(f"Known vendors: {n_vendors}")
print(f"\n{'Strategy':<25} {'Vendors flagged':>16} {'Vendor flag rate':>17}")
print("-" * 62)
for name, r in results.items():
    vendor_flagged = (r["flagged"] & vendor_mask).sum()
    rate = vendor_flagged / n_vendors if n_vendors > 0 else 0
    print(f"{name:<25} {vendor_flagged:>16} {rate:>16.1%}")

In [ ]:
## --- Pairwise overlap between strategies ---
## How much do the flagged sets agree with each other?
## High overlap between global and a grouped strategy = the grouping isn't adding much.

strat_names = list(results.keys())
overlap = pd.DataFrame(index=strat_names, columns=strat_names, dtype=float)
for a in strat_names:
    for b in strat_names:
        set_a = set(df.index[results[a]["flagged"] == 1])
        set_b = set(df.index[results[b]["flagged"] == 1])
        union = len(set_a | set_b)
        overlap.loc[a, b] = len(set_a & set_b) / union if union > 0 else 1.0

print("Jaccard overlap between flagged sets:")
print(overlap.round(2).to_string())

In [ ]:
## --- Per-feature variance reduction by strategy ---
## Which features actually benefit from grouping?

vr_rows = []
for feat in z_features:
    row = {"feature": feat}
    global_std = df[feat].std()
    for name, group_col in strategies.items():
        if group_col is None:
            row[name] = 0.0
            continue
        gsizes = df.groupby(group_col).size()
        valid = gsizes[gsizes >= MIN_GROUP].index
        grp_stds = df[df[group_col].isin(valid)].groupby(group_col)[feat].std()
        row[name] = (1 - grp_stds.mean() / global_std) if global_std > 0 and len(grp_stds) > 0 else 0.0
    vr_rows.append(row)

vr_df = pd.DataFrame(vr_rows).set_index("feature")
print("Per-feature variance reduction (higher = grouping helps more):\n")
print(vr_df.round(3).to_string())